# Telepathy-like teaching of a small agent model

Companion notebook to the paper-track writeup **"Copy, Correct, or Rewrite? Telepathy-Like Teaching of a Small Agent Model and What Keeps It From Forgetting"** (Gemma 4 Developer Agent competition). Code, data and the pre-registration log: https://github.com/syn-ack-ai/copy-correct-rewrite

**The idea.** A *state corrector* is a small network that writes a correction into a student model's hidden states. Fed the teacher's hidden states, it makes the student predict what the teacher would. In *telepathy-like teaching* this corrected student, not the teacher's output, is the target for a LoRA on the student, online, one item at a time. Afterwards only the LoRA is kept: no teacher, no corrector. Teacher: Gemma 4 E4B. Student: Gemma 4 E2B.

**What this notebook does.** It clones the public repository at a pinned commit and rebuilds, from the released result files only (CPU, about a minute):

1. **Table 1**: the online teachers on one GPU type.
2. **What the teacher's states add**: students of the cross-model corrector against students of the same corrector fed only the student's own states, paired on 240 held rows.
3. **Figure 2**: closeness to the teacher over the stream, and what each ingredient adds.
4. **The live test**: LoRAs on 80 held-out agent tasks in the competition's harness, with a null-adapter control.

Training cannot be rerun here (its inputs are private: agent rows recorded in the competition harness and text corpora). Everything below reads the per-run files that the training sessions wrote.

Teachers of the LoRA: **C** E4B's own distribution; **D** the student corrected by a corrector fed E4B's hidden states (telepathy-like); **S** the same corrector form fed the student's own states (the control); **D4 / S4** correctors fitted four times longer; **DI** a cross-model corrector penalised for writes the student's states cannot predict.

In [ ]:
import json, subprocess, sys
from pathlib import Path
REPO, COMMIT = "https://github.com/syn-ack-ai/copy-correct-rewrite", "2eb861342c9ec9443782e09181bd968cb48b5e64"
ROOT = Path("/tmp/copy-correct-rewrite")          # outside /kaggle/working, so the clone is not saved as notebook output
if not ROOT.exists():
    subprocess.run(["git", "clone", "-q", REPO, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", COMMIT], check=True)
print("repository at", subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
R = json.load(open(ROOT / "data/results.json"))["online g4"]
def run(*args):
    p = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, cwd=ROOT); print(p.stdout); p.returncode and print(p.stderr[-2000:])
print("teachers:", ", ".join(f"{k} (seeds {', '.join(sorted(v))})" for k, v in R["table"].items()))
print("gate partition:", R["partition"])

## 1. Table 1: the online teachers on one GPU type

*Held CE*: cross-entropy to E4B on held repositories at the end of the 600-item stream (untrained E2B: 0.825; lower is closer). *Prequential gain*: nats per token gained over the original E2B on each item before learning it (higher is faster). *Net*: of 220 held decisions, those fixed (of the 52 where E2B alone differs from E4B) minus those broken (of the other 168).

In [ ]:
name = {"C": "C  E4B's distribution", "S": "S  self-corrector", "D": "D  cross-model corrector", "DI": "DI cross-model, imitable", "S4": "S4 self-corrector, 4x", "D4": "D4 cross-model, 4x"}
print(f"{'teacher':28s} {'seed':>4s} {'held CE':>8s} {'gain':>7s} {'fixed':>6s} {'broken':>7s} {'net':>5s} {'reaches C final at item':>24s}")
for k in ("C", "S", "D", "DI", "S4", "D4"):
    for sd, v in sorted(R["table"].get(k, {}).items()):
        print(f"{name[k]:28s} {sd:>4s} {v['held ce']:8.4f} {v['gain']:7.4f} {v['fixed']:6d} {v['broken']:7d} {v['net']:+5d} {str(v.get("item reaching C's final", '')):>24s}")

The same table straight from the per-run files, with the paired sign tests on the 220 decisions (`scripts/g4_table.py`):

In [ ]:
run(ROOT / "scripts/g4_table.py")

## 2. What the teacher's states add

Every student is scored on the same 240 held gate rows (118 episodes, 17 repositories). Differences are first minus second, so negative means the first student is closer to E4B. Intervals are bootstraps over rows, episodes and repositories.

This paired analysis was added after two seeds; the bar set in advance (0.005 lower in every seed) was not met. See `PREREGISTRATION.md` in the repository: addendum 206 and amendments 206c and 206d.

In [ ]:
run(ROOT / "scripts/g4_rows.py")

In [ ]:
P = R["pairs"]; K = "rows: mean, lo, hi, repos closer, repos"
cs = P["cross vs self"]
print(f"cross-model minus self-corrector, {cs['pairs']} pairs: {cs[K][0]:+.4f} nats [{cs[K][1]:+.4f}, {cs[K][2]:+.4f}], closer in {cs[K][3]} of {cs[K][4]} repositories")
print("pairs with the cross-model student closer:", cs["pairs with the cross student closer"], "of", cs["pairs"], "| pair means:", cs["pair means"])
print("per decision (more cross students match E4B, more self students match, p):", cs["decisions: more cross students match, more self students match, p"])
for k in ("D vs C", "S vs C", "D4 vs D", "S4 vs S", "DI vs D"):
    v = P[k]["all seeds"]; print(f"for scale, {k}: {v[K][0]:+.4f} [{v[K][1]:+.4f}, {v[K][2]:+.4f}] (seeds {v['seeds']})")

## 3. Figure 2

In [ ]:
from IPython.display import Image, display
out = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
subprocess.run([sys.executable, str(ROOT / "scripts/figures.py"), str(ROOT / "data/results.json"), str(out), "fig_online"], check=True, capture_output=True)
display(Image(filename=str(out / "fig7_online.png"), width=1000))

## 4. The live test

Each attempt runs 80 held-out-repository agent tasks once in the competition's harness (E4B solved none of them). The *null adapter* is D's adapter with its output matrices set to zero: untrained E2B through the same serving path. Comparisons are by task on the solve rate, with exact two-sided sign tests, as pre-registered.

In [ ]:
run(ROOT / "scripts/live_table.py")

## Where to look next

- The paper: [`paper.md`](https://github.com/syn-ack-ai/copy-correct-rewrite/blob/main/paper.md). Which pre-registered predictions held: [`SCORECARD.md`](https://github.com/syn-ack-ai/copy-correct-rewrite/blob/main/SCORECARD.md).
- Designs, predictions and results as they were written: [`PREREGISTRATION.md`](https://github.com/syn-ack-ai/copy-correct-rewrite/blob/main/PREREGISTRATION.md).
- One result file per run: `data/online/` (`g4_*` are the one-GPU table; `a100_*` and unprefixed files are earlier runs on other GPU types).
- The online test: `code/transfer/online.py`; one session: `code/colab/xq_online.sh`.